# SDT Policy Polarization Study — Data Cleaning Pipeline
**Updated: April 2026 (Full Study, 3-of-4 Design)**

This notebook performs all data cleaning, variable recoding, theme-order extraction,
sample representativeness assessment, and SDT diagnostic checks for the full study.

### Pipeline overview
1. Load & standardize raw Qualtrics export
2. Drop metadata rows, merge duplicate timer columns
3. Recode demographics (age, gender, education, race, income, hispanic)
4. Recode political variables (ideology, party ID 7-pt, news, political interest)
5. Score political knowledge (3 items) and numeracy (5 items)
6. Recode risk measures (general risk-taking + Holt–Laury CRRA)
7. Recode policy preferences & compute composite scales
8. Recode base-rate belief questions (3-of-4 design per theme)
9. Recode error threshold questions & compute log-ratio
10. Extract theme presentation order from Qualtrics flow metadata
11. Derive experimental condition variables (B_before_T, BT_before_P)
12. Assess sample representativeness vs. U.S. benchmarks
13. Diagnose 3-of-4 SDT question draw coverage
14. Export cleaned dataset

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
from scipy import stats
import re
import warnings
warnings.filterwarnings('ignore')

# ===================== CONFIGURATION =====================

DATA_PATH = "/content/drive/MyDrive/Projects/SDT/Data/rawdata.csv"   # <-- UPDATE to your raw Qualtrics CSV path
OUTPUT_PATH = "/content/drive/MyDrive/Projects/SDT/Data/cleaned_data.csv"

THEMES = ['disease', 'armed', 'conv', 'welfare', 'immi', 'vote', 'air', 'firearm', 'auto']

QUESTION_TYPES = ['tp', 'fp', 'tn', 'fn']

# Qualtrics raw theme prefixes → standard internal names
THEME_RAW_TO_STANDARD = {
    'disease': 'disease', 'armed': 'armed',
    'crime': 'conv', 'conv': 'conv',
    'walfare': 'welfare', 'welfare': 'welfare',
    'immigrant': 'immi', 'immi': 'immi',
    'vote': 'vote', 'air': 'air',
    'firearm': 'firearm', 'auto': 'auto',
}

# Political direction for aligned log-ratio (positive = conservative)
POLITICAL_DIRECTION = {
    'disease': 0, 'armed': 1, 'conv': 1, 'welfare': -1,
    'immi': 1, 'vote': 1, 'air': 0, 'firearm': -1, 'auto': 0
}

# DO column names (Qualtrics display-order metadata for each theme's base-rate block)
DO_COLUMNS = {
    'disease': 'Baselinequestions-diseasescreening_DO',
    'armed':   'Baselinequestions-Armedconflict_DO',
    'conv':    'Baselinequestions-Convictions_DO',
    'welfare': 'Baselinequestions-Welfarebenefits_DO',
    'immi':    'Baselinequestions-Immigration_DO',
    'vote':    'Baselinequestions-Voting_DO',
    'air':     'Baselinequestions-Airportsecurity_DO',
    'firearm': 'Baselinequestions-firearmchecks_DO',
    'auto':    'Baselinequestions-Auto-driving_DO',
}

# Theme display names for the flow order columns
THEME_DISPLAY_NAMES = {
    'diseasescreening': 'disease', 'Armedconflict': 'armed',
    'Convictions': 'conv', 'Welfarebenefits': 'welfare',
    'Immigration': 'immi', 'Voting': 'vote',
    'Airportsecurity': 'air', 'firearmchecks': 'firearm',
    'Auto-driving': 'auto',
}

print("Configuration loaded.")

Configuration loaded.


## 1. Load Raw Data

In [3]:
df = pd.read_csv(DATA_PATH, dtype=str)
print(f"Raw data: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"First 10 columns: {list(df.columns[:10])}")

Raw data: 728 rows × 337 columns
First 10 columns: ['StartDate', 'EndDate', 'Status', 'IPAddress', 'Progress', 'Duration (in seconds)', 'Finished', 'RecordedDate', 'ResponseId', 'RecipientLastName']


## 2. Drop Qualtrics Metadata Rows & Standardize Columns

In [4]:
# Drop Qualtrics header rows (question text / ImportId rows)
rows_before = len(df)
rows_to_drop = []

if len(df) >= 1:
    first_row = df.iloc[0].astype(str)
    if first_row.str.len().mean() > 50 or first_row.str.contains('ImportId', na=False).any():
        rows_to_drop.append(0)
if len(df) >= 2:
    second_row = df.iloc[1].astype(str)
    if second_row.str.contains('ImportId', na=False).any():
        rows_to_drop.append(1)

if rows_to_drop:
    df = df.drop(index=rows_to_drop).reset_index(drop=True)
    print(f"Dropped {len(rows_to_drop)} Qualtrics metadata row(s). Rows: {rows_before} → {len(df)}")
else:
    print("No metadata rows detected.")

Dropped 2 Qualtrics metadata row(s). Rows: 728 → 726


In [5]:
# --- Column Standardization ---
# Handles: theme name variations, base-rate column format, risk attitude columns,
# error threshold naming, timer columns

rename_map = {}

for col in df.columns:
    new_col = col

    # 1. Base-rate data columns: {raw_theme}_{TYPE}_1 → base_{std_theme}_{type}
    m = re.match(r'^(disease|armed|crime|conv|walfare|welfare|immigrant|immi|vote|air|firearm|auto)_(TP|FP|TN|FN)_1$', col)
    if m:
        raw_theme, qtype = m.group(1), m.group(2).lower()
        std_theme = THEME_RAW_TO_STANDARD.get(raw_theme.lower(), raw_theme.lower())
        new_col = f'base_{std_theme}_{qtype}'

    # 2. Risk attitude columns: risk2_intro_N → risk_attitude_N
    m2 = re.match(r'^risk2_intro_(\d+)$', col)
    if m2:
        new_col = f'risk_attitude_{m2.group(1)}'

    # 3. Error threshold: err_binary → err_disease_binary (known Qualtrics naming error)
    if col == 'err_binary':
        new_col = 'err_disease_binary'

    # 4. Timer standardization: remove '_timer' infix pattern variations
    #    e.g., base_armed_intro_First Click → base_armed_timer_First Click (keep as-is per codebook)

    if new_col != col:
        rename_map[col] = new_col

if rename_map:
    df = df.rename(columns=rename_map)
    print(f"Renamed {len(rename_map)} columns:")
    for old, new in sorted(rename_map.items()):
        print(f"  {old} → {new}")
else:
    print("No columns needed renaming.")

Renamed 47 columns:
  air_FN_1 → base_air_fn
  air_FP_1 → base_air_fp
  air_TN_1 → base_air_tn
  air_TP_1 → base_air_tp
  armed_FN_1 → base_armed_fn
  armed_FP_1 → base_armed_fp
  armed_TN_1 → base_armed_tn
  armed_TP_1 → base_armed_tp
  auto_FN_1 → base_auto_fn
  auto_FP_1 → base_auto_fp
  auto_TN_1 → base_auto_tn
  auto_TP_1 → base_auto_tp
  crime_FN_1 → base_conv_fn
  crime_FP_1 → base_conv_fp
  crime_TN_1 → base_conv_tn
  crime_TP_1 → base_conv_tp
  disease_FN_1 → base_disease_fn
  disease_FP_1 → base_disease_fp
  disease_TN_1 → base_disease_tn
  disease_TP_1 → base_disease_tp
  err_binary → err_disease_binary
  firearm_FN_1 → base_firearm_fn
  firearm_FP_1 → base_firearm_fp
  firearm_TN_1 → base_firearm_tn
  firearm_TP_1 → base_firearm_tp
  immigrant_FN_1 → base_immi_fn
  immigrant_FP_1 → base_immi_fp
  immigrant_TN_1 → base_immi_tn
  immigrant_TP_1 → base_immi_tp
  risk2_intro_1 → risk_attitude_1
  risk2_intro_10 → risk_attitude_10
  risk2_intro_2 → risk_attitude_2
  risk2_intro_

In [6]:
# Merge duplicate timer/metadata columns (Q-numbered duplicates → named columns)
def copy_if_empty(df, src_prefix, dst_prefix,
                  fields=("First Click", "Last Click", "Page Submit", "Click Count")):
    for f in fields:
        src_col, dst_col = f"{src_prefix}_{f}", f"{dst_prefix}_{f}"
        if src_col not in df.columns or dst_col not in df.columns:
            continue
        dst_empty = df[dst_col].isna() | df[dst_col].astype(str).str.strip().eq("")
        df.loc[dst_empty, dst_col] = df.loc[dst_empty, src_col]

merge_pairs = [
    ("Q203", "base_disease"), ("Q201", "err_disease"),
    ("Q229", "base_armed"),   ("Q86",  "err_armed"),
    ("Q242", "base_conv"),    ("Q182", "err_conv"),
    ("Q254", "base_welfare"), ("Q183", "err_welfare"),
    ("Q264", "base_immi"),    ("Q184", "err_immi"),
    ("Q282", "base_vote"),    ("Q185", "err_vote"),
    ("Q292", "base_air"),     ("Q186", "err_air"),
    ("Q300", "base_firearm"), ("Q187", "err_firearm"),
    ("Q308", "base_auto"),    ("Q188", "err_auto"),
]

for src, dst in merge_pairs:
    copy_if_empty(df, src, dst)

# Drop Q-numbered timer duplicates and other unused columns
q_timer_cols = []
for src, _ in merge_pairs:
    for suffix in ["First Click", "Last Click", "Page Submit", "Click Count"]:
        q_timer_cols.append(f"{src}_{suffix}")

misc_drop = [
    'RecipientLastName', 'RecipientFirstName', 'RecipientEmail', 'ExternalReference',
    'mid', 'comments',
    'Q43_First Click', 'Q43_Last Click', 'Q43_Page Submit', 'Q43_Click Count',
    'Q60_First Click', 'Q60_Last Click', 'Q60_Page Submit', 'Q60_Click Count',
]

# Drop display-order columns that are not needed downstream (but keep Baselinequestions_DO columns)
do_drop = [c for c in df.columns if re.match(r'^(risk_taking_DO|pknow\d+_DO|Q\d+_DO)$', c)]
# NOTE: FL_*_DO columns are kept here — they are consumed by theme_order extraction (cell 23) and dropped there

all_drop = q_timer_cols + misc_drop + do_drop
df = df.drop(columns=[c for c in all_drop if c in df.columns], errors='ignore')

# Coerce Page Submit columns to float
for c in [c for c in df.columns if c.endswith('Page Submit')]:
    df[c] = pd.to_numeric(df[c].astype(str).str.replace(',', ''), errors='coerce')

print(f"After merges and drops: {df.shape[1]} columns remaining.")

After merges and drops: 330 columns remaining.


## 3. Recode Demographics

In [7]:
# --- AGE ---
df['age'] = pd.to_numeric(df['age'], errors='coerce')
n_invalid_age = df['age'].isna().sum()
n_under18 = (df['age'] < 18).sum()
df = df[df['age'].notna() & (df['age'] >= 18)].copy()
print(f"Age: dropped {n_invalid_age} invalid + {n_under18} under-18. Remaining: {len(df)}")

# --- MOBILE ---
df['mobile_bin'] = df['mobile'].map({'Yes': 1, 'No': 0})

# --- GENDER ---
df['gender'] = df['gender'].astype(str).str.strip().str.lower()
df = df[~df['gender'].isin(["", "nan", "na", "-99"])].copy()
df['gender_clean'] = df['gender'].map({
    'female': 'Female', 'woman': 'Female', 'f': 'Female',
    'male': 'Male', 'man': 'Male', 'm': 'Male'
}).fillna('Other')
print(f"Gender distribution:\n{df['gender_clean'].value_counts().to_string()}")

# --- EDUCATION ---
df['edu.7'] = df['edu.7'].astype(str).str.strip()
df = df[~df['edu.7'].str.lower().isin(["", "nan", "na", "-99"])].copy()
edu_map = {
    "Did not graduate from high school": 1,
    "High school diploma or the equivalent (GED)": 2,
    "Some college": 3, "Associate degree": 4,
    "Bachelor's degree": 5, "Master's degree": 6,
    "Professional or doctorate degree": 7
}
df['education_num'] = df['edu.7'].map(edu_map)
print(f"\nEducation distribution:\n{df['education_num'].value_counts().sort_index().to_string()}")

# --- HISPANIC ---
df['hispanic'] = df['hispan'].map({'Yes': 1, 'No': 0}) if 'hispan' in df.columns else np.nan

# --- RACE ---
if 'race' in df.columns:
    df['race_list'] = df['race'].fillna("").apply(lambda x: [i.strip() for i in x.split(",") if i.strip()])
    def clean_race(races):
        if len(races) == 0: return np.nan
        if len(races) > 1: return "Multiracial"
        return races[0]
    df['race_clean'] = df['race_list'].apply(clean_race)
    df = df.drop(columns=['race_list'])
    print(f"\nRace distribution:\n{df['race_clean'].value_counts().to_string()}")

# --- INCOME (new for full study) ---
if 'income' in df.columns:
    income_map = {
        "Less than $15,000": 1, "$15,000–$29,999": 2, "$30,000–$49,999": 3,
        "$50,000–$74,999": 4, "$75,000–$99,999": 5, "$100,000–$149,999": 6,
        "$150,000–$199,999": 7, "$200,000 or more": 8, "Prefer not to say": np.nan
    }
    # Handle Qualtrics encoding of en-dash vs hyphen
    df['income_raw'] = df['income'].astype(str).str.strip()
    df['income_raw'] = df['income_raw'].str.replace('\u2013', '–')  # normalize en-dash
    df['income_num'] = df['income_raw'].map(income_map)
    # Fallback: try fuzzy matching on dollar amounts
    for idx in df.index[df['income_num'].isna() & df['income_raw'].notna()]:
        val = df.loc[idx, 'income_raw']
        if 'prefer' in val.lower():
            df.loc[idx, 'income_num'] = np.nan
        elif '200' in val:
            df.loc[idx, 'income_num'] = 8
        elif '150' in val:
            df.loc[idx, 'income_num'] = 7
        elif '100' in val:
            df.loc[idx, 'income_num'] = 6
        elif '75' in val:
            df.loc[idx, 'income_num'] = 5
        elif '50' in val:
            df.loc[idx, 'income_num'] = 4
        elif '30' in val:
            df.loc[idx, 'income_num'] = 3
        elif '15' in val and 'less' not in val.lower():
            df.loc[idx, 'income_num'] = 2
        elif 'less' in val.lower():
            df.loc[idx, 'income_num'] = 1
    print(f"\nIncome distribution:\n{df['income_num'].value_counts().sort_index().to_string()}")

print(f"\nAfter demographics cleaning: {len(df)} rows")

Age: dropped 2 invalid + 1 under-18. Remaining: 723
Gender distribution:
gender_clean
Male      364
Female    355
Other       3

Education distribution:
education_num
1     10
2     97
3    115
4     61
5    289
6    128
7     22

Race distribution:
race_clean
White                               479
Black or African American           121
Asian/Pacific Islander               59
Multi-racial                         21
Multiracial                          18
Other                                15
American Indian or Alaska Native      6
-99                                   3

Income distribution:
income_num
1.0     37
2.0     58
3.0     83
4.0    128
5.0    121
6.0    106
7.0     51
8.0     35

After demographics cleaning: 722 rows


## 4. Recode Political Variables

In [8]:
# --- IDEOLOGY (7-point) ---
df['ideo.7'] = df['ideo.7'].astype(str).str.strip().str.lower()
df = df[~df['ideo.7'].isin(["", "nan", "na", "-99"])].copy()

ideology_map = {
    "extremely liberal": 1, "liberal": 2, "slightly liberal": 3,
    "moderate; middle of the road": 4, "moderate": 4,
    "slightly conservative": 5, "conservative": 6, "extremely conservative": 7
}
df['ideology_num'] = df['ideo.7'].map(ideology_map)
print(f"Ideology distribution:\n{df['ideology_num'].value_counts().sort_index().to_string()}")

# --- PARTY ID (7-point, constructed from branching) ---
for col in ['pid.4', 'strong.dem', 'strong.rep', 'lean']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.lower()

df = df[~df['pid.4'].isin(["", "nan", "na", "-99"])].copy()

def recode_pid7(row):
    pid = row.get('pid.4', '')
    if pid == "democrat":
        s = row.get('strong.dem', '')
        if s == "strong democrat": return 1
        elif s == "not very strong democrat": return 2
    elif pid == "republican":
        s = row.get('strong.rep', '')
        if s == "strong republican": return 7
        elif s == "not very strong republican": return 6
    elif pid in ["independent", "something else"]:
        lean = row.get('lean', '')
        if lean == "closer to the democratic party": return 3
        elif lean == "closer to the republican party": return 5
        else: return 4
    return np.nan

df['party_id_7'] = df.apply(recode_pid7, axis=1)
print(f"\nParty ID 7-point:\n{df['party_id_7'].value_counts().sort_index().to_string()}")

# --- NEWS CONSUMPTION (ordinal 1-6) ---
if 'news' in df.columns:
    news_map = {
        "Never": 1, "Less than once a week": 2, "1–2 days a week": 3,
        "3–4 days a week": 4, "5–6 days a week": 5, "Every day": 6
    }
    # Also try with en-dash variants
    df['news_raw'] = df['news'].astype(str).str.strip().str.replace('\u2013', '–')
    df['news_num'] = df['news_raw'].map(news_map)
    # Fallback numeric
    df['news_num'] = df['news_num'].fillna(pd.to_numeric(df['news'].astype(str).str.strip(), errors='coerce'))
    print(f"\nNews consumption:\n{df['news_num'].value_counts().sort_index().to_string()}")

# --- POLITICAL INTEREST (ordinal 1-5) ---
if 'p_interest' in df.columns:
    pint_map = {
        "Not at all interested": 1, "Slightly interested": 2,
        "Moderately interested": 3, "Very interested": 4, "Extremely interested": 5
    }
    df['p_interest_num'] = df['p_interest'].astype(str).str.strip().map(pint_map)
    df['p_interest_num'] = df['p_interest_num'].fillna(pd.to_numeric(df['p_interest'].astype(str).str.strip(), errors='coerce'))
    print(f"\nPolitical interest:\n{df['p_interest_num'].value_counts().sort_index().to_string()}")

print(f"\nAfter political variable cleaning: {len(df)} rows")

Ideology distribution:
ideology_num
1    108
2    157
3     78
4    149
5     77
6    107
7     46

Party ID 7-point:
party_id_7
1    182
2    114
3     89
4     86
5     54
6     85
7    112

News consumption:
news_num
-99.0      2
 1.0      10
 2.0      44
 3.0      80
 4.0     156
 5.0     123
 6.0     220

Political interest:
p_interest_num
-99.0      3
 1.0      21
 2.0     108
 3.0     228
 4.0     195
 5.0      80

After political variable cleaning: 722 rows


## 5. Score Political Knowledge (3 items) & Numeracy (5 items)

In [9]:
# --- POLITICAL KNOWLEDGE (updated: 3 items, not 6) ---
# pknow1: "The Supreme Court"
# pknow2: "Two"
# pknow3: "Four years"

pknow_correct = {
    'pknow1': 'The Supreme Court',
    'pknow2': 'Two',
    'pknow3': 'Four years',
}

pknow_score_parts = []
for col, answer in pknow_correct.items():
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()
        correct = df[col].str.lower() == answer.lower()
        df[f'{col}_correct'] = correct.astype(int)
        pknow_score_parts.append(f'{col}_correct')
        print(f"{col}: {correct.sum()} correct ({100*correct.mean():.1f}%)")

df['pknow_score'] = df[pknow_score_parts].sum(axis=1) if pknow_score_parts else 0
print(f"\nPknow score distribution:\n{df['pknow_score'].value_counts().sort_index().to_string()}")

# --- NUMERACY (5 items) ---
# num1=500, num2=10, num3=20, num4=100, num5=0.1 or "0.1%"
num_answers = {'num1': 500, 'num2': 10, 'num3': 20, 'num4': 100}
num_parts = []

for col, ans in num_answers.items():
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[f'{col}_correct'] = (df[col] == ans).astype(int)
        num_parts.append(f'{col}_correct')

if 'num5' in df.columns:
    df['num5_correct'] = df['num5'].astype(str).str.strip().isin(['0.1', '0.1%']).astype(int)
    num_parts.append('num5_correct')

df['num_correct'] = df[num_parts].sum(axis=1) if num_parts else 0
print(f"\nNumeracy score distribution:\n{df['num_correct'].value_counts().sort_index().to_string()}")

pknow1: 531 correct (73.5%)
pknow2: 425 correct (58.9%)
pknow3: 601 correct (83.2%)

Pknow score distribution:
pknow_score
0     97
1     75
2    168
3    382

Numeracy score distribution:
num_correct
0     22
1     25
2     41
3     95
4    293
5    246


## 6. Recode Risk Measures

In [10]:
# --- GENERAL RISK-TAKING (6 items, Likert 1-5) ---
risk_cols = [f'risk_taking_{i}' for i in range(1, 7)]
valid_risk = {"never": 1, "rarely": 2, "somewhat often": 3, "very often": 4, "all the time": 5}

for col in risk_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.lower()
        df[col + '_num'] = df[col].map(valid_risk)

risk_num_cols = [c + '_num' for c in risk_cols if c + '_num' in df.columns]
df['risk_taking_mean'] = df[risk_num_cols].mean(axis=1)
df['risk_taking_sum'] = df[risk_num_cols].sum(axis=1)

# Risk attention flag (see codebook appendix)
# Flag if all items identical or other suspicious patterns
if len(risk_num_cols) > 0:
    df['risk_attention_flag'] = 0  # placeholder — refine based on appendix criteria
    # Simple heuristic: flag if SD across items is 0 (all same response)
    df.loc[df[risk_num_cols].std(axis=1) == 0, 'risk_attention_flag'] = 1
    print(f"Risk attention flags: {df['risk_attention_flag'].sum()} flagged")

# --- HOLT-LAURY LOTTERY TASK (10 items) ---
risk_att_cols = [f'risk_attitude_{i}' for i in range(1, 11)]
hl_map = {3: 0, 4: 1}  # 3=Option A (safe), 4=Option B (risky)

for col in risk_att_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col + '_num'] = df[col].map(hl_map)

hl_num_cols = [c + '_num' for c in risk_att_cols if c + '_num' in df.columns]

if hl_num_cols:
    df['risk_attitude_sum'] = df[hl_num_cols].sum(axis=1)
    df['risk_attitude_mean'] = df[hl_num_cols].mean(axis=1)

    # Compute switch point: first row where participant chose risky (=1)
    def find_switch_point(row):
        for i, col in enumerate(hl_num_cols, 1):
            if row[col] == 1:
                return i
        return 11  # never switched
    df['risk_switch_point'] = df.apply(find_switch_point, axis=1)

    # CRRA estimate (midpoint of interval)
    crra_intervals = {
        1: -1.50, 2: -0.95, 3: -0.49, 4: -0.14, 5: 0.15,
        6: 0.41, 7: 0.68, 8: 0.97, 9: 1.37, 10: 1.73, 11: 2.00
    }
    df['risk_crra_estimate'] = df['risk_switch_point'].map(crra_intervals)
    print(f"\nSwitch point distribution:\n{df['risk_switch_point'].value_counts().sort_index().to_string()}")

print(f"\nRisk measures recoded. Rows: {len(df)}")

Risk attention flags: 77 flagged

Switch point distribution:
risk_switch_point
1      75
2      35
3      21
4      21
5     114
6     103
7     115
8      86
9      51
10     31
11     70

Risk measures recoded. Rows: 722


## 7. Recode Policy Preferences & Composite Scales

In [11]:
# Mapping dictionaries
five_point = {
    "Strongly oppose": 1, "Somewhat oppose": 2, "Not sure/no opinion": 3,
    "Somewhat favor": 4, "Strongly favor": 5,
    "Very bad thing": 5, "Mostly bad thing": 4, "Mostly good thing": 2, "Very good thing": 1,
}
civlib_map = {
    "Strongly oppose": 1, "Somewhat oppose": 2, "Not sure/no opinion": 3,
    "Somewhat support": 4, "Strongly support": 5,
}
guns_map = {"More strict": 1, "Kept the same": 2, "Less strict": 3}
crime_map = {"Too tough": 1, "About right": 2, "Not tough enough": 3}
hawk_map = {
    "Strongly disagree": 1, "Somewhat disagree": 2, "Not sure/no opinion": 3,
    "Somewhat agree": 4, "Strongly agree": 5,
}
hawk_reverse = {
    "Strongly agree": 1, "Somewhat agree": 2, "Not sure/no opinion": 3,
    "Somewhat disagree": 4, "Strongly disagree": 5,
}

# Apply recoding
pref_recode = {
    'pref_crime_rec':    ('pref_crime', crime_map),
    'pref_work_rec':     ('pref_work', five_point),
    'pref_id_rec':       ('pref_id', five_point),
    'pref_imm_rec':      ('pref_imm', five_point),
    'pref_civlib_1_rec': ('pref_civlib_1', civlib_map),
    'pref_civlib_2_rec': ('pref_civlib_2', civlib_map),
    'pref_civlib_3_rec': ('pref_civlib_3', civlib_map),
    'pref_civlib_4_rec': ('pref_civlib_4', civlib_map),
    'pref_guns_rec':     ('pref_guns', guns_map),
    'pref_hawk_1_rec':   ('pref_hawk_raw_1', hawk_map),
    'pref_hawk_2_rec':   ('pref_hawk_raw_2', hawk_map),
    'pref_hawk_3_rec':   ('pref_hawk_raw_3', hawk_map),
    'pref_hawk_4_rec':   ('pref_hawk_raw_4_REVERSED', hawk_reverse),
}

for new_col, (raw_col, mapping) in pref_recode.items():
    if raw_col in df.columns:
        df[new_col] = df[raw_col].map(mapping)

# Composite scales
df['civlib_scale'] = df[['pref_civlib_1_rec','pref_civlib_2_rec',
                         'pref_civlib_3_rec','pref_civlib_4_rec']].mean(axis=1)
df['hawk_scale'] = df[['pref_hawk_1_rec','pref_hawk_2_rec',
                       'pref_hawk_3_rec','pref_hawk_4_rec']].mean(axis=1)
df['policy_conservatism'] = df[['pref_id_rec','pref_imm_rec','pref_guns_rec',
                                 'pref_work_rec','pref_crime_rec',
                                 'civlib_scale','hawk_scale']].mean(axis=1)

# 0-1 rescaling
def rescale_01(s, lo, hi): return (s - lo) / (hi - lo)
for col in ['civlib_scale','hawk_scale','policy_conservatism']:
    if col in df.columns:
        df[col + '_01'] = rescale_01(df[col], 1, 5)
for col in ['pref_crime_rec','pref_guns_rec']:
    if col in df.columns:
        df[col + '_01'] = rescale_01(df[col], 1, 3)

print("Policy preference variables recoded.")
print(f"Policy conservatism: mean={df['policy_conservatism'].mean():.2f}, "
      f"sd={df['policy_conservatism'].std():.2f}")

Policy preference variables recoded.
Policy conservatism: mean=2.59, sd=0.67


## 8. Recode Base-Rate Beliefs (3-of-4 Design)

Each participant receives **3 of 4** possible SDT question types (FP, FN, TP, TN) per domain.
The 4 possible draw configurations are: {FP,FN,TP}, {FP,FN,TN}, {FP,TP,TN}, {FN,TP,TN}.

Every configuration guarantees:
- At least one signal-present question (TP or FN) **and** one signal-absent question (FP or TN) → SDT always computable
- At least one consistency-checkable pair (FP+TN or FN+TP) → consistency always assessable

In [12]:
def safe_numeric(val):
    """Convert to float; return None for missing/invalid."""
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return None
    s = str(val).strip()
    if s in ('', 'nan', 'NA', '-99'):
        return None
    try:
        return float(s)
    except ValueError:
        return None

def is_answered(val):
    return safe_numeric(val) is not None

# Process each theme
for theme in THEMES:
    prefix = f'base_{theme}_'

    # Identify which questions were answered
    answered_lists = []
    for idx in df.index:
        answered = []
        for qtype in QUESTION_TYPES:
            col = f'{prefix}{qtype}'
            if col in df.columns and is_answered(df.loc[idx, col]):
                answered.append(qtype)
        answered_lists.append(answered)

    df[f'{theme}_questions_answered'] = ['|'.join(a) if a else 'none' for a in answered_lists]
    df[f'{theme}_n_answered'] = [len(a) for a in answered_lists]

    # Convert base-rate columns to numeric
    for qtype in QUESTION_TYPES:
        col = f'{prefix}{qtype}'
        if col in df.columns:
            df[col] = df[col].apply(lambda v: safe_numeric(v))

    # Classify pair type (3-of-4 design: identify which question is missing)
    def classify_3of4(answered):
        s = set(answered)
        if len(s) == 0: return 'no_questions'
        if len(s) == 1: return 'single_question'
        if len(s) == 2: return 'two_questions'
        if len(s) == 4: return 'all_four'
        # Exactly 3: identify which is missing
        missing = set(QUESTION_TYPES) - s
        if len(missing) == 1:
            return f'three_questions_missing_{list(missing)[0]}'
        return 'unknown'

    df[f'{theme}_pair_type'] = [classify_3of4(a) for a in answered_lists]

    # Consistency check: FP+TN and/or FN+TP
    def check_consistency(row):
        fp_col, tn_col = f'{prefix}fp', f'{prefix}tn'
        fn_col, tp_col = f'{prefix}fn', f'{prefix}tp'
        sums = []
        if fp_col in df.columns and tn_col in df.columns:
            fp_v, tn_v = safe_numeric(row.get(fp_col)), safe_numeric(row.get(tn_col))
            if fp_v is not None and tn_v is not None:
                sums.append(fp_v + tn_v)
        if fn_col in df.columns and tp_col in df.columns:
            fn_v, tp_v = safe_numeric(row.get(fn_col)), safe_numeric(row.get(tp_col))
            if fn_v is not None and tp_v is not None:
                sums.append(fn_v + tp_v)
        return sums

    consistency_sums = df.apply(check_consistency, axis=1)
    df[f'{theme}_can_consistency'] = consistency_sums.apply(lambda x: len(x) > 0)
    # Use first available consistency pair
    df[f'{theme}_consistency_sum'] = consistency_sums.apply(
        lambda x: x[0] if len(x) > 0 else np.nan)
    df[f'{theme}_consistency_deviation'] = (df[f'{theme}_consistency_sum'] - 1000).abs()

    # SDT computability: need at least one signal-present (TP or FN) and one signal-absent (FP or TN)
    def can_compute_sdt(answered):
        s = set(answered)
        has_signal_present = bool(s & {'tp', 'fn'})
        has_signal_absent = bool(s & {'fp', 'tn'})
        return has_signal_present and has_signal_absent

    df[f'{theme}_can_sdt'] = [can_compute_sdt(a) for a in answered_lists]

    # Compute FPR and TPR
    def compute_fpr(row):
        fp_col, tn_col = f'{prefix}fp', f'{prefix}tn'
        fp_v = safe_numeric(row.get(fp_col)) if fp_col in df.columns else None
        tn_v = safe_numeric(row.get(tn_col)) if tn_col in df.columns else None
        if fp_v is not None:
            return fp_v / 1000.0
        elif tn_v is not None:
            return (1000 - tn_v) / 1000.0
        return np.nan

    def compute_tpr(row):
        tp_col, fn_col = f'{prefix}tp', f'{prefix}fn'
        tp_v = safe_numeric(row.get(tp_col)) if tp_col in df.columns else None
        fn_v = safe_numeric(row.get(fn_col)) if fn_col in df.columns else None
        if tp_v is not None:
            return tp_v / 1000.0
        elif fn_v is not None:
            return (1000 - fn_v) / 1000.0
        return np.nan

    df[f'{theme}_fpr'] = df.apply(compute_fpr, axis=1)
    df[f'{theme}_tpr'] = df.apply(compute_tpr, axis=1)

    # Compute d-prime and criterion c (with Haldane-Anscombe correction)
    def compute_sdt(row):
        fpr = row[f'{theme}_fpr']
        tpr = row[f'{theme}_tpr']
        if pd.isna(fpr) or pd.isna(tpr):
            return np.nan, np.nan
        # Clip extreme values
        fpr = np.clip(fpr, 0.001, 0.999)
        tpr = np.clip(tpr, 0.001, 0.999)
        z_fpr = stats.norm.ppf(fpr)
        z_tpr = stats.norm.ppf(tpr)
        d_prime = z_tpr - z_fpr
        c = -0.5 * (z_tpr + z_fpr)
        return d_prime, c

    sdt_results = df.apply(compute_sdt, axis=1, result_type='expand')
    df[f'{theme}_d_prime'] = sdt_results[0]
    df[f'{theme}_c'] = sdt_results[1]

# Aggregate across themes
df['total_themes_answered'] = sum(
    (df[f'{t}_n_answered'] > 0).astype(int) for t in THEMES)
df['total_questions_answered'] = sum(
    df[f'{t}_n_answered'] for t in THEMES)
df['themes_with_sdt'] = sum(
    df[f'{t}_can_sdt'].astype(int) for t in THEMES)
df['themes_with_consistency'] = sum(
    df[f'{t}_can_consistency'].astype(int) for t in THEMES)

print("Base-rate beliefs recoded (3-of-4 design).")
print(f"  Mean themes answered: {df['total_themes_answered'].mean():.1f}")
print(f"  Mean questions answered: {df['total_questions_answered'].mean():.1f}")
print(f"  Mean themes with SDT: {df['themes_with_sdt'].mean():.1f}")
print(f"  Mean themes with consistency: {df['themes_with_consistency'].mean():.1f}")

Base-rate beliefs recoded (3-of-4 design).
  Mean themes answered: 6.8
  Mean questions answered: 19.3
  Mean themes with SDT: 6.3
  Mean themes with consistency: 6.2


## 9. Recode Error Thresholds & Compute Log-Ratio

In [13]:
# =====================================================================
# Error Threshold Recoding
# =====================================================================
# Each theme has three raw columns:
#   err_{theme}_binary  — text of which error type chosen as more acceptable
#   err_{theme}_fp      — numeric X value if FP chosen
#   err_{theme}_fn      — numeric X value if FN chosen
#
# Choice detection strategy:
#   PRIMARY: If err_{theme}_fp has data → participant chose FP.
#            If err_{theme}_fn has data → participant chose FN.
#   (Qualtrics only shows the fp slider if FP was chosen, fn slider if FN.)
#   FALLBACK: If both or neither have data, parse the binary text column.

def parse_err_binary_text(val):
    """Fallback parser for the binary choice text column.
    Looks for 'fp' or 'fn' substring in the corresponding column name
    is not possible here — we match on question content instead.
    Returns 'FN' or 'FP' or None."""
    if pd.isna(val) or str(val).strip() in ('', 'nan', 'NA'):
        return None
    val_str = str(val).strip()

    # Try numeric coding (1=FN, 4=FP per codebook)
    try:
        num = int(float(val_str))
        if num == 1: return 'FN'
        elif num == 4: return 'FP'
    except (ValueError, TypeError):
        pass

    # Text fallback: not used as primary, only as last resort
    return None

# Diagnostic: show binary column values for first theme
diag_col = 'err_disease_binary'
if diag_col in df.columns:
    unique_vals = df[diag_col].dropna().unique()
    print(f"Diagnostic — {diag_col} unique values ({len(unique_vals)}):")
    for v in unique_vals[:10]:
        print(f"  '{v}'")

for theme in THEMES:
    prefix = f'err_{theme}'
    binary_col = f'{prefix}_binary'
    fp_col = f'{prefix}_fp'
    fn_col = f'{prefix}_fn'

    # Initialize output columns
    df[f'{theme}_err_choice'] = None
    df[f'{theme}_err_value'] = np.nan
    df[f'{theme}_err_ratio'] = np.nan
    df[f'{theme}_err_log_ratio'] = np.nan
    df[f'{theme}_err_assigned'] = False

    if binary_col not in df.columns:
        print(f"  Warning: {binary_col} not found")
        continue

    for idx in df.index:
        binary_val = df.loc[idx, binary_col] if binary_col in df.columns else None
        fp_val = safe_numeric(df.loc[idx, fp_col]) if fp_col in df.columns else None
        fn_val = safe_numeric(df.loc[idx, fn_col]) if fn_col in df.columns else None

        # Check if theme was assigned (any non-missing data)
        has_binary = not (pd.isna(binary_val) or str(binary_val).strip() in ('', 'nan'))
        has_data = has_binary or fp_val is not None or fn_val is not None
        if not has_data:
            continue

        df.loc[idx, f'{theme}_err_assigned'] = True

        # PRIMARY: determine choice from which slider column has data
        if fp_val is not None and fn_val is None:
            choice = 'FP'
        elif fn_val is not None and fp_val is None:
            choice = 'FN'
        elif fp_val is not None and fn_val is not None:
            # Both filled (shouldn't happen) — fall back to binary column
            choice = parse_err_binary_text(binary_val)
        else:
            # Neither slider has data but binary was answered — parse text
            choice = parse_err_binary_text(binary_val)

        df.loc[idx, f'{theme}_err_choice'] = choice

        # Compute log-ratio based on choice
        if choice == 'FP' and fp_val is not None and fp_val > 0:
            df.loc[idx, f'{theme}_err_value'] = fp_val
            df.loc[idx, f'{theme}_err_ratio'] = fp_val       # FP:FN = X:1
            df.loc[idx, f'{theme}_err_log_ratio'] = np.log10(fp_val)
        elif choice == 'FN' and fn_val is not None and fn_val > 0:
            df.loc[idx, f'{theme}_err_value'] = fn_val
            df.loc[idx, f'{theme}_err_ratio'] = 1.0 / fn_val  # FP:FN = 1:X
            df.loc[idx, f'{theme}_err_log_ratio'] = -np.log10(fn_val)

# Aligned log-ratio (positive = conservative direction)
for theme, direction in POLITICAL_DIRECTION.items():
    lr_col = f'{theme}_err_log_ratio'
    aligned_col = f'{theme}_err_log_ratio_aligned'
    if lr_col in df.columns:
        if direction == -1:
            df[aligned_col] = -df[lr_col]
        elif direction == 1:
            df[aligned_col] = df[lr_col]
        else:
            df[aligned_col] = np.nan

# Summary statistics
assigned_cols = [f'{t}_err_assigned' for t in THEMES]
log_ratio_cols = [f'{t}_err_log_ratio' for t in THEMES]
choice_cols = [f'{t}_err_choice' for t in THEMES]

df['n_err_themes_assigned'] = df[assigned_cols].sum(axis=1)
df['n_err_valid'] = df[log_ratio_cols].notna().sum(axis=1)
df['err_log_ratio_mean'] = df[log_ratio_cols].mean(axis=1)
df['err_log_ratio_sd'] = df[log_ratio_cols].std(axis=1)
df['n_err_fp_chosen'] = df[choice_cols].apply(lambda row: (row == 'FP').sum(), axis=1)
df['n_err_fn_chosen'] = df[choice_cols].apply(lambda row: (row == 'FN').sum(), axis=1)
df['err_prop_fp_chosen'] = df['n_err_fp_chosen'] / (df['n_err_fp_chosen'] + df['n_err_fn_chosen'])

print("\nError thresholds recoded.")
for theme in THEMES:
    n_assigned = df[f'{theme}_err_assigned'].sum()
    n_valid = df[f'{theme}_err_log_ratio'].notna().sum()
    n_fp = (df[f'{theme}_err_choice'] == 'FP').sum()
    n_fn = (df[f'{theme}_err_choice'] == 'FN').sum()
    n_none = n_assigned - n_fp - n_fn
    print(f"  {theme:10s}: assigned={n_assigned}, FP={n_fp}, FN={n_fn}, "
          f"unparsed={n_none}, valid log_ratio={n_valid}")

Diagnostic — err_disease_binary unique values (2):
  'A person infected with a dangerous disease incorrectly tests negative for it.'
  'A person not infected with a dangerous disease incorrectly tests positive for it.'

Error thresholds recoded.
  disease   : assigned=524, FP=311, FN=72, unparsed=141, valid log_ratio=383
  armed     : assigned=539, FP=61, FN=310, unparsed=168, valid log_ratio=371
  conv      : assigned=544, FP=120, FN=320, unparsed=104, valid log_ratio=440
  welfare   : assigned=573, FP=126, FN=308, unparsed=139, valid log_ratio=434
  immi      : assigned=553, FP=204, FN=213, unparsed=136, valid log_ratio=417
  vote      : assigned=535, FP=194, FN=249, unparsed=92, valid log_ratio=443
  air       : assigned=557, FP=285, FN=50, unparsed=222, valid log_ratio=335
  firearm   : assigned=559, FP=335, FN=67, unparsed=157, valid log_ratio=402
  auto      : assigned=549, FP=283, FN=58, unparsed=208, valid log_ratio=341


## 10. Extract Theme Presentation Order

Qualtrics FL_*_DO columns encode the flow order. We extract which themes were presented
and in what sequence, producing:
- `theme_order`: pipe-separated list of theme names in presentation order
- `theme_{name}_position`: 1-indexed position for each theme (NaN if not assigned)

In [14]:
# Extract theme order from FL_*_DO metadata columns
# FL_*_DO columns contain pipe-delimited flow block names.
# We look for entries starting with 'Baselinequestions-' to extract theme order.

fl_do_columns = [col for col in df.columns if re.match(r'^FL_\d+_DO$', col)]
print(f"Found {len(fl_do_columns)} FL metadata columns")

# Also check: are there Baselinequestions-*_DO columns directly?
# These are per-theme display-order columns (separate from FL flow columns)
bq_do_columns = [col for col in df.columns if col.startswith('Baselinequestions-') and col.endswith('_DO')]
print(f"Found {len(bq_do_columns)} Baselinequestions_DO columns")

theme_orders = []
for idx in df.index:
    # Concatenate all FL_*_DO values
    combined_text = ''
    for col in fl_do_columns:
        val = df.loc[idx, col]
        if pd.notna(val) and str(val).strip():
            combined_text += '|' + str(val)

    # Split by pipes, tabs, and spaces
    parts = re.split(r'[|\t\s]+', combined_text)

    baseline_order = []
    for part in parts:
        part = part.strip()
        if 'Baselinequestions-' in part and '_DO' not in part:
            clean = part.replace('Baselinequestions-', '').strip()
            std_name = THEME_DISPLAY_NAMES.get(clean, clean)
            if std_name not in baseline_order:
                baseline_order.append(std_name)
    theme_orders.append(baseline_order)

df['theme_order'] = ['|'.join(order) if order else '' for order in theme_orders]

# Create position columns for each theme
for theme in THEMES:
    positions = []
    for order in theme_orders:
        if theme in order:
            positions.append(order.index(theme) + 1)  # 1-indexed
        else:
            positions.append(np.nan)
    df[f'theme_{theme}_position'] = positions

# Drop FL_*_DO columns (no longer needed)
df = df.drop(columns=fl_do_columns, errors='ignore')

# Summary
n_with_order = (df['theme_order'] != '').sum()
mean_themes = np.mean([len(o) for o in theme_orders if len(o) > 0]) if any(o for o in theme_orders) else 0
print(f"Theme order extracted for {n_with_order}/{len(df)} participants")
print(f"Mean themes in order: {mean_themes:.1f}")
if n_with_order > 0:
    print(f"\nSample theme orders (first 5):")
    shown = 0
    for order in df['theme_order']:
        if order:
            print(f"  {order}")
            shown += 1
            if shown >= 5:
                break
else:
    # Diagnostic: show what's in the FL columns
    print("\nDiagnostic — no theme orders found. FL_*_DO column samples:")
    # Check if FL columns were actually available
    print(f"  fl_do_columns found: {len(fl_do_columns)}")
    # Check Baselinequestions_DO columns as alternative source
    if bq_do_columns:
        print(f"  Baselinequestions_DO columns can indicate which themes were assigned:")
        for col in bq_do_columns:
            n_nonempty = df[col].dropna().astype(str).str.strip().ne('').sum()
            print(f"    {col}: {n_nonempty} non-empty")

Found 39 FL metadata columns
Found 9 Baselinequestions_DO columns
Theme order extracted for 707/722 participants
Mean themes in order: 6.9

Sample theme orders (first 5):
  armed
  disease|conv|welfare|vote|firearm
  disease|welfare|immi|vote|firearm
  armed|conv|welfare|immi|vote|air|auto
  disease|armed|conv|welfare|immi|firearm|auto


## 11. Derive Experimental Condition Variables

In [15]:
# Derive B_before_T and BT_before_P from the 'order' column
if 'order' in df.columns:
    df['order'] = df['order'].astype(str).str.strip().str.upper()

    # B_before_T: 1 if base-rate preceded threshold (BTP, PBT conditions)
    df['B_before_T'] = df['order'].map({
        'BTP': 1, 'PBT': 1, 'TBP': 0, 'PTB': 0
    })

    # BT_before_P: 1 if B+T modules preceded P block (BTP, TBP conditions)
    df['BT_before_P'] = df['order'].map({
        'BTP': 1, 'TBP': 1, 'PBT': 0, 'PTB': 0
    })

    print(f"Experimental conditions:")
    print(f"  Order distribution:\n{df['order'].value_counts().to_string()}")
    print(f"\n  B_before_T:\n{df['B_before_T'].value_counts().to_string()}")
    print(f"\n  BT_before_P:\n{df['BT_before_P'].value_counts().to_string()}")
else:
    print("WARNING: 'order' column not found. Cannot derive experimental conditions.")
    print("  Available columns containing 'order':", [c for c in df.columns if 'order' in c.lower()])

Experimental conditions:
  Order distribution:
order
PBT    202
BTP    177
TBP    170
PTB    163
NAN     10

  B_before_T:
B_before_T
1.0    379
0.0    333

  BT_before_P:
BT_before_P
0.0    365
1.0    347


## 11b. Data Quality & Attention Checks

Five independent quality signals, each producing a binary flag.
A participant is flagged as low-quality if **2 or more** flags are raised.

| Flag | Logic | Rationale |
|------|-------|-----------|
| `flag_hl_irrational` | >1 switch between A→B in Holt-Laury | Rational EUT: at most one switch point |
| `flag_acceleration` | Median total theme time (base + err) drops >50% from first half to second half of the 7-theme sequence | Accelerating across themes = fatigue / loss of attention |
| `flag_recaptcha` | `Q_RecaptchaScore < 0.5` | Bot detection (Qualtrics reCAPTCHA v3) |
| `flag_too_fast` | Median total theme time < 5 seconds | Impossibly fast responding |
| `flag_too_slow` | Median total theme time > 300 seconds (5 min) | Likely AFK / distracted |
| `is_incomplete` | Fewer than 7 of 9 themes answered | Didn't complete the study |

**Unit of analysis for timing flags:** Total time per theme = base-rate Page Submit + error threshold
Page Submit. Acceleration is evaluated theme-by-theme in presentation order across the 7
assigned themes, NOT between base-rate vs. error-threshold pages within a theme.

**Robustness:** Timer columns may be missing for some themes (Qualtrics platform issues).
The acceleration check uses whatever theme-level totals are available (minimum 4 needed).
Theme presentation order is extracted from `theme_order`; if unavailable, themes are
ordered by their position columns.

In [16]:
# =====================================================================
# DATA QUALITY FLAG SYSTEM
# =====================================================================

# --- FLAG 1: Holt-Laury Irrational Switching ---
# Rational behavior: monotonic — at most ONE switch from A(=3) to B(=4).
# Multiple switches indicate random clicking or misunderstanding.

risk_att_cols_raw = [f'risk_attitude_{i}' for i in range(1, 11)]
hl_cols_present = [c for c in risk_att_cols_raw if c in df.columns]

def count_hl_switches(row):
    """Count A→B and B→A transitions in the Holt-Laury sequence."""
    choices = []
    for col in hl_cols_present:
        val = pd.to_numeric(row[col], errors='coerce')
        if val in (3, 4):
            choices.append(val)
    if len(choices) < 2:
        return 0
    return sum(1 for i in range(1, len(choices)) if choices[i] != choices[i-1])

df['hl_n_switches'] = df.apply(count_hl_switches, axis=1)
df['flag_hl_irrational'] = (df['hl_n_switches'] > 1).astype(int)
print(f"Flag 1 — HL irrational (>1 switch): {df['flag_hl_irrational'].sum()} flagged "
      f"({100*df['flag_hl_irrational'].mean():.1f}%)")
print(f"  Switch count distribution: {df['hl_n_switches'].value_counts().sort_index().to_dict()}")

# --- FLAG 2: Acceleration (declining THEME-LEVEL total time) ---
# Each theme occupies two survey pages: base-rate questions and error threshold.
# Total theme time = base Page Submit + err Page Submit.
# We get the theme totals in presentation order, then compare whether the
# participant sped up from the first half to the second half of their 7-theme sequence.

# Step 2a: Discover timer columns
# Raw Qualtrics names vary: base_crime_timer_Page Submit, base_armed_intro_Page Submit, etc.
_THEME_RAW_VARIANTS = {
    'disease': ['disease'], 'armed': ['armed'], 'conv': ['conv', 'crime'],
    'welfare': ['welfare', 'walfare'], 'immi': ['immi', 'immigrant'],
    'vote': ['vote'], 'air': ['air'], 'firearm': ['firearm'], 'auto': ['auto'],
}

def _find_timer_col(section, theme, columns):
    """Find the Page Submit timer column for a section ('base'/'err') + theme."""
    col_set = set(columns)
    for raw_name in _THEME_RAW_VARIANTS.get(theme, [theme]):
        for mid in ['_timer_', '_intro_', '_']:
            candidate = f'{section}_{raw_name}{mid}Page Submit'
            if candidate in col_set:
                return candidate
    return None

_base_timer = {}
_err_timer = {}
for t in THEMES:
    bc = _find_timer_col('base', t, df.columns)
    ec = _find_timer_col('err', t, df.columns)
    if bc: _base_timer[t] = bc
    if ec: _err_timer[t] = ec

print(f"\nTimer columns found:")
for t in THEMES:
    b = _base_timer.get(t, '(none)')
    e = _err_timer.get(t, '(none)')
    print(f"  {t:10s}: base={b},  err={e}")
n_with_any_timer = sum(1 for t in THEMES if t in _base_timer or t in _err_timer)
print(f"  Themes with at least one timer: {n_with_any_timer}/9")

# Step 2b: Compute total time per theme per participant
def get_theme_total_time(row, theme):
    """Sum base + err Page Submit for one theme. Returns None if no timer available."""
    total = 0.0
    found = False
    for timer_map in [_base_timer, _err_timer]:
        col = timer_map.get(theme)
        if col:
            val = pd.to_numeric(row.get(col), errors='coerce')
            if pd.notna(val) and val > 0:
                total += val
                found = True
    return total if found else None

# Step 2c: Get theme totals in presentation order
def get_ordered_theme_totals(row):
    """Get total time per theme in presentation order."""
    # Try theme_order first (pipe-separated string from cell 23)
    order_str = row.get('theme_order', '')
    if order_str and pd.notna(order_str) and str(order_str).strip():
        themes_in_order = [t.strip() for t in str(order_str).split('|') if t.strip()]
    else:
        # Fallback: use theme position columns, sorted by position
        theme_positions = []
        for t in THEMES:
            pos_col = f'theme_{t}_position'
            if pos_col in row.index:
                pos = pd.to_numeric(row.get(pos_col), errors='coerce')
                if pd.notna(pos):
                    theme_positions.append((int(pos), t))
        theme_positions.sort()
        themes_in_order = [t for _, t in theme_positions]

    # Collect total times for each theme in order
    totals = []
    for theme in themes_in_order:
        t = get_theme_total_time(row, theme)
        if t is not None:
            totals.append(t)
    return totals

theme_totals = df.apply(get_ordered_theme_totals, axis=1)

def check_acceleration(totals):
    """Flag if second-half median theme time is <50% of first-half median."""
    if len(totals) < 4:
        return 0
    mid = len(totals) // 2
    first_half = np.median(totals[:mid])
    second_half = np.median(totals[mid:])
    if first_half <= 0:
        return 0
    return 1 if (second_half / first_half) < 0.5 else 0

df['flag_acceleration'] = theme_totals.apply(check_acceleration)
n_with_times = theme_totals.apply(len).gt(0).sum()
print(f"\nFlag 2 — Acceleration (2nd-half themes <50% of 1st-half): "
      f"{df['flag_acceleration'].sum()} flagged ({100*df['flag_acceleration'].mean():.1f}%)")
print(f"  Participants with theme-level timing: {n_with_times}/{len(df)}")
if n_with_times > 0:
    themes_per = theme_totals.apply(len)
    print(f"  Themes with timing per person: mean={themes_per[themes_per > 0].mean():.1f}, "
          f"median={themes_per[themes_per > 0].median():.0f}")
    all_flat = [t for ts in theme_totals if ts for t in ts]
    if all_flat:
        print(f"  Per-theme total time (sec): median={np.median(all_flat):.1f}, "
              f"mean={np.mean(all_flat):.1f}, min={np.min(all_flat):.1f}, max={np.max(all_flat):.1f}")

# --- FLAG 3: reCAPTCHA bot detection ---
recaptcha_col = 'Q_RecaptchaScore'
if recaptcha_col in df.columns:
    df[recaptcha_col] = pd.to_numeric(df[recaptcha_col], errors='coerce')
    df['flag_recaptcha'] = ((df[recaptcha_col] < 0.5) & df[recaptcha_col].notna()).astype(int)
    print(f"\nFlag 3 — reCAPTCHA (<0.5): {df['flag_recaptcha'].sum()} flagged "
          f"({100*df['flag_recaptcha'].mean():.1f}%)")
    print(f"  Score distribution: mean={df[recaptcha_col].mean():.2f}, "
          f"median={df[recaptcha_col].median():.2f}")
else:
    df['flag_recaptcha'] = 0
    print(f"\nFlag 3 — reCAPTCHA: column '{recaptcha_col}' not found, flag set to 0 for all")

# --- FLAGS 4 & 5: Too fast / Too slow (based on per-theme totals) ---
median_theme_time = theme_totals.apply(lambda t: np.median(t) if len(t) > 0 else np.nan)
df['median_theme_time'] = median_theme_time

df['flag_too_fast'] = ((median_theme_time < 5) & median_theme_time.notna()).astype(int)
print(f"\nFlag 4 — Too fast (median theme total <5s): {df['flag_too_fast'].sum()} flagged "
      f"({100*df['flag_too_fast'].mean():.1f}%)")

df['flag_too_slow'] = ((median_theme_time > 300) & median_theme_time.notna()).astype(int)
print(f"\nFlag 5 — Too slow (median theme total >300s): {df['flag_too_slow'].sum()} flagged "
      f"({100*df['flag_too_slow'].mean():.1f}%)")

# --- INCOMPLETE FLAG (separate from attention) ---
df['is_incomplete'] = (df['total_themes_answered'] < 7)
print(f"\nIncomplete (<7 themes): {df['is_incomplete'].sum()} "
      f"({100*df['is_incomplete'].mean():.1f}%)")

# =====================================================================
# COMPOSITE QUALITY LABEL: flag ≥2 of 5 attention flags → low quality
# =====================================================================
attention_flag_cols = ['flag_hl_irrational', 'flag_acceleration', 'flag_recaptcha',
                       'flag_too_fast', 'flag_too_slow']
df['n_attention_flags'] = df[attention_flag_cols].sum(axis=1)
df['low_quality'] = (df['n_attention_flags'] >= 2).astype(int)

# Combined filter: passes if not low_quality AND not incomplete
df['passes_attention'] = ((df['low_quality'] == 0) & (~df['is_incomplete'])).astype(int)

print(f"\n{'='*60}")
print(f"QUALITY SUMMARY")
print(f"{'='*60}")
print(f"  Total participants: {len(df)}")
print(f"  Attention flag distribution:")
for n_flags in sorted(df['n_attention_flags'].unique()):
    count = (df['n_attention_flags'] == n_flags).sum()
    print(f"    {int(n_flags)} flags: {count} ({100*count/len(df):.1f}%)")
print(f"  Low quality (≥2 flags): {df['low_quality'].sum()} ({100*df['low_quality'].mean():.1f}%)")
print(f"  Incomplete: {df['is_incomplete'].sum()} ({100*df['is_incomplete'].mean():.1f}%)")
print(f"  Passes all filters: {df['passes_attention'].sum()} ({100*df['passes_attention'].mean():.1f}%)")
print(f"\n  Co-occurrence matrix (attention flags):")
flag_cooccur = df[attention_flag_cols].corr().round(2)
print(flag_cooccur.to_string())

Flag 1 — HL irrational (>1 switch): 105 flagged (14.5%)
  Switch count distribution: {0: 93, 1: 524, 2: 13, 3: 28, 4: 16, 5: 19, 6: 14, 7: 6, 8: 3, 9: 6}

Timer columns found:
  disease   : base=base_disease_timer_Page Submit,  err=err_disease_timer_Page Submit
  armed     : base=base_armed_timer_Page Submit,  err=err_armed_timer_Page Submit
  conv      : base=base_crime_timer_Page Submit,  err=err_conv_timer_Page Submit
  welfare   : base=base_welfare_timer_Page Submit,  err=err_welfare_timer_Page Submit
  immi      : base=base_immi_timer_Page Submit,  err=err_immi_timer_Page Submit
  vote      : base=base_vote_timer_Page Submit,  err=err_vote_timer_Page Submit
  air       : base=base_air_timer_Page Submit,  err=err_air_timer_Page Submit
  firearm   : base=base_firearm_timer_Page Submit,  err=err_firearm_timer_Page Submit
  auto      : base=base_auto_timer_Page Submit,  err=err_auto_timer_Page Submit
  Themes with at least one timer: 9/9

Flag 2 — Acceleration (2nd-half themes <50% of

## 12. Sample Representativeness Assessment

Compare sample demographics to U.S. population benchmarks (Census Bureau / CPS / ANES).
Benchmarks are approximate adult (18+) population shares.

In [17]:
# =====================================================================
# U.S. ADULT POPULATION BENCHMARKS (Census / CPS / ANES approximations)
# =====================================================================

us_benchmarks = {
    'gender': {
        'Female': 0.512, 'Male': 0.488,
    },
    'race': {
        'White': 0.596, 'Black or African American': 0.124,
        'Asian/Pacific Islander': 0.062, 'Multiracial': 0.044,
        'American Indian or Alaska Native': 0.013, 'Other': 0.161,
    },
    'hispanic': {0: 0.808, 1: 0.192},
    'education': {
        1: 0.11, 2: 0.27, 3: 0.20, 4: 0.09,
        5: 0.21, 6: 0.09, 7: 0.04,
    },
    'income': {
        1: 0.09, 2: 0.09, 3: 0.12, 4: 0.14,
        5: 0.12, 6: 0.15, 7: 0.07, 8: 0.11,
    },
    'party_id': {
        'Dem (1-3)': 0.47, 'Independent (4)': 0.07, 'Rep (5-7)': 0.46,
    },
    'age_group': {
        '18-29': 0.21, '30-44': 0.25, '45-64': 0.32, '65+': 0.22,
    },
}

def print_representativeness(df):
    print("=" * 75)
    print("SAMPLE REPRESENTATIVENESS ASSESSMENT (vs. U.S. Adult Population)")
    print("=" * 75)

    # --- Gender ---
    print("\n--- GENDER ---")
    if 'gender_clean' in df.columns:
        sample_gender = df['gender_clean'].value_counts(normalize=True)
        for cat, us_pct in us_benchmarks['gender'].items():
            s_pct = sample_gender.get(cat, 0)
            diff = s_pct - us_pct
            flag = " ⚠️" if abs(diff) > 0.10 else ""
            print(f"  {cat:30s}: Sample={s_pct:.1%}  U.S.={us_pct:.1%}  Diff={diff:+.1%}{flag}")

    # --- Race ---
    print("\n--- RACE ---")
    if 'race_clean' in df.columns:
        sample_race = df['race_clean'].value_counts(normalize=True)
        for cat, us_pct in us_benchmarks['race'].items():
            s_pct = sample_race.get(cat, 0)
            diff = s_pct - us_pct
            flag = " ⚠️" if abs(diff) > 0.10 else ""
            print(f"  {cat:40s}: Sample={s_pct:.1%}  U.S.={us_pct:.1%}  Diff={diff:+.1%}{flag}")

    # --- Hispanic ---
    print("\n--- HISPANIC/LATINO ---")
    if 'hispanic' in df.columns:
        sample_hisp = df['hispanic'].value_counts(normalize=True)
        for cat, us_pct in us_benchmarks['hispanic'].items():
            s_pct = sample_hisp.get(cat, 0)
            diff = s_pct - us_pct
            flag = " ⚠️" if abs(diff) > 0.10 else ""
            label = "Hispanic" if cat == 1 else "Non-Hispanic"
            print(f"  {label:30s}: Sample={s_pct:.1%}  U.S.={us_pct:.1%}  Diff={diff:+.1%}{flag}")

    # --- Education ---
    print("\n--- EDUCATION ---")
    if 'education_num' in df.columns:
        edu_labels = {1:'< HS', 2:'HS diploma', 3:'Some college', 4:'Associate',
                      5:'Bachelor\'s', 6:'Master\'s', 7:'Prof/Doctorate'}
        sample_edu = df['education_num'].value_counts(normalize=True)
        for cat, us_pct in us_benchmarks['education'].items():
            s_pct = sample_edu.get(cat, 0)
            diff = s_pct - us_pct
            flag = " ⚠️" if abs(diff) > 0.10 else ""
            print(f"  {edu_labels.get(cat, str(cat)):30s}: Sample={s_pct:.1%}  U.S.={us_pct:.1%}  Diff={diff:+.1%}{flag}")

    # --- Income ---
    print("\n--- HOUSEHOLD INCOME ---")
    if 'income_num' in df.columns:
        inc_labels = {1:'<$15k', 2:'$15-30k', 3:'$30-50k', 4:'$50-75k',
                      5:'$75-100k', 6:'$100-150k', 7:'$150-200k', 8:'$200k+'}
        sample_inc = df['income_num'].dropna().astype(int).value_counts(normalize=True)
        for cat, us_pct in us_benchmarks['income'].items():
            s_pct = sample_inc.get(cat, 0)
            diff = s_pct - us_pct
            flag = " ⚠️" if abs(diff) > 0.10 else ""
            print(f"  {inc_labels.get(cat, str(cat)):30s}: Sample={s_pct:.1%}  U.S.={us_pct:.1%}  Diff={diff:+.1%}{flag}")

    # --- Age groups ---
    print("\n--- AGE GROUPS ---")
    if 'age' in df.columns:
        bins = [18, 30, 45, 65, 200]
        labels = ['18-29', '30-44', '45-64', '65+']
        df['_age_group'] = pd.cut(df['age'], bins=bins, labels=labels, right=False)
        sample_age = df['_age_group'].value_counts(normalize=True)
        for cat, us_pct in us_benchmarks['age_group'].items():
            s_pct = sample_age.get(cat, 0)
            diff = s_pct - us_pct
            flag = " ⚠️" if abs(diff) > 0.10 else ""
            print(f"  {cat:30s}: Sample={s_pct:.1%}  U.S.={us_pct:.1%}  Diff={diff:+.1%}{flag}")
        df = df.drop(columns=['_age_group'])

    # --- Party ID ---
    print("\n--- PARTY IDENTIFICATION ---")
    if 'party_id_7' in df.columns:
        pid = df['party_id_7'].dropna()
        groups = {
            'Dem (1-3)': pid.isin([1,2,3]).sum() / len(pid),
            'Independent (4)': pid.isin([4]).sum() / len(pid),
            'Rep (5-7)': pid.isin([5,6,7]).sum() / len(pid),
        }
        for cat, s_pct in groups.items():
            us_pct = us_benchmarks['party_id'].get(cat, 0)
            diff = s_pct - us_pct
            flag = " ⚠️" if abs(diff) > 0.10 else ""
            print(f"  {cat:30s}: Sample={s_pct:.1%}  U.S.≈{us_pct:.1%}  Diff={diff:+.1%}{flag}")

    print("\n" + "=" * 75)
    print("Note: ⚠️ flags indicate >10 percentage point deviation from benchmark.")
    print("Prolific samples typically skew younger, more educated, more liberal")
    print("relative to the general U.S. adult population.")
    print("=" * 75)

print_representativeness(df)

SAMPLE REPRESENTATIVENESS ASSESSMENT (vs. U.S. Adult Population)

--- GENDER ---
  Female                        : Sample=49.2%  U.S.=51.2%  Diff=-2.0%
  Male                          : Sample=50.4%  U.S.=48.8%  Diff=+1.6%

--- RACE ---
  White                                   : Sample=66.3%  U.S.=59.6%  Diff=+6.7%
  Black or African American               : Sample=16.8%  U.S.=12.4%  Diff=+4.4%
  Asian/Pacific Islander                  : Sample=8.2%  U.S.=6.2%  Diff=+2.0%
  Multiracial                             : Sample=2.5%  U.S.=4.4%  Diff=-1.9%
  American Indian or Alaska Native        : Sample=0.8%  U.S.=1.3%  Diff=-0.5%
  Other                                   : Sample=2.1%  U.S.=16.1%  Diff=-14.0% ⚠️

--- HISPANIC/LATINO ---
  Non-Hispanic                  : Sample=88.8%  U.S.=80.8%  Diff=+8.0%
  Hispanic                      : Sample=11.2%  U.S.=19.2%  Diff=-8.0%

--- EDUCATION ---
  < HS                          : Sample=1.4%  U.S.=11.0%  Diff=-9.6%
  HS diploma            

## 13. Diagnose 3-of-4 SDT Question Draw Coverage

This section assesses the randomized base-rate question assignment:
1. **Draw distribution**: Are the 4 possible 3-of-4 configurations balanced across participants?
2. **SDT computability**: What proportion of participant×theme observations support d'/c computation?
3. **Consistency checkability**: What proportion support numeric consistency checks?
4. **Dual coverage**: With 3-of-4, every normal draw should enable *both* SDT and consistency.

In [18]:
print("=" * 75)
print("3-OF-4 SDT QUESTION DRAW DIAGNOSTICS")
print("=" * 75)

# 1. Pair type distribution per theme
print("\n--- PAIR TYPE DISTRIBUTION PER THEME ---")
for theme in THEMES:
    pt_col = f'{theme}_pair_type'
    if pt_col in df.columns:
        dist = df[pt_col].value_counts()
        n_three = dist.filter(like='three_questions').sum()
        n_total = len(df[df[f'{theme}_n_answered'] > 0])
        print(f"\n  {theme.upper()} (n={n_total} with data):")
        for val, count in dist.items():
            print(f"    {val:35s}: {count:4d}  ({100*count/len(df):.1f}%)")

# 2. Draw configuration balance across participants
print("\n\n--- DRAW CONFIGURATION BALANCE ---")
config_counts = {}
for theme in THEMES:
    pt_col = f'{theme}_pair_type'
    if pt_col in df.columns:
        for config in ['three_questions_missing_fp', 'three_questions_missing_fn',
                       'three_questions_missing_tp', 'three_questions_missing_tn']:
            config_counts.setdefault(config, 0)
            config_counts[config] += (df[pt_col] == config).sum()

total_draws = sum(config_counts.values())
if total_draws > 0:
    print(f"  Total 3-of-4 draws across all themes: {total_draws}")
    expected_each = total_draws / 4
    for config, count in sorted(config_counts.items()):
        pct = 100 * count / total_draws
        missing_type = config.replace('three_questions_missing_', '').upper()
        deviation = abs(pct - 25.0)
        flag = " ⚠️ unbalanced" if deviation > 5 else ""
        print(f"  Missing {missing_type}: {count:5d}  ({pct:.1f}%, expected ~25%){flag}")

# 3. SDT computability
print("\n\n--- SDT COMPUTABILITY ---")
for theme in THEMES:
    n_assigned = (df[f'{theme}_n_answered'] > 0).sum()
    n_sdt = df[f'{theme}_can_sdt'].sum() if f'{theme}_can_sdt' in df.columns else 0
    pct = 100 * n_sdt / n_assigned if n_assigned > 0 else 0
    flag = " ⚠️" if pct < 95 else " ✓"
    print(f"  {theme:10s}: {n_sdt}/{n_assigned} ({pct:.1f}%) can compute d'/c{flag}")

# 4. Consistency checkability
print("\n--- CONSISTENCY CHECKABILITY ---")
for theme in THEMES:
    n_assigned = (df[f'{theme}_n_answered'] > 0).sum()
    n_con = df[f'{theme}_can_consistency'].sum() if f'{theme}_can_consistency' in df.columns else 0
    pct = 100 * n_con / n_assigned if n_assigned > 0 else 0
    flag = " ⚠️" if pct < 95 else " ✓"
    print(f"  {theme:10s}: {n_con}/{n_assigned} ({pct:.1f}%) can check consistency{flag}")

# 5. Dual coverage (both SDT + consistency)
print("\n--- DUAL COVERAGE (SDT + Consistency per participant×theme) ---")
dual_total, dual_both = 0, 0
for theme in THEMES:
    sdt_col = f'{theme}_can_sdt'
    con_col = f'{theme}_can_consistency'
    if sdt_col in df.columns and con_col in df.columns:
        assigned_mask = df[f'{theme}_n_answered'] > 0
        n = assigned_mask.sum()
        n_both = (df[sdt_col] & df[con_col] & assigned_mask).sum()
        dual_total += n
        dual_both += n_both
if dual_total > 0:
    print(f"  Overall: {dual_both}/{dual_total} ({100*dual_both/dual_total:.1f}%) have both SDT + consistency")
    print(f"  (3-of-4 design guarantees 100% dual coverage for normal draws)")

# 6. Consistency deviation summary
print("\n--- CONSISTENCY DEVIATION SUMMARY ---")
all_devs = []
for theme in THEMES:
    dev_col = f'{theme}_consistency_deviation'
    if dev_col in df.columns:
        vals = df[dev_col].dropna()
        all_devs.extend(vals.tolist())
        if len(vals) > 0:
            print(f"  {theme:10s}: median={vals.median():.0f}, mean={vals.mean():.0f}, "
                  f"≤50={100*(vals <= 50).mean():.0f}%, >100={100*(vals > 100).mean():.0f}%")

if all_devs:
    all_devs = pd.Series(all_devs)
    print(f"\n  OVERALL: median={all_devs.median():.0f}, mean={all_devs.mean():.0f}, "
          f"≤50={100*(all_devs <= 50).mean():.0f}%, >100={100*(all_devs > 100).mean():.0f}%")

print("\n" + "=" * 75)

3-OF-4 SDT QUESTION DRAW DIAGNOSTICS

--- PAIR TYPE DISTRIBUTION PER THEME ---

  DISEASE (n=521 with data):
    no_questions                       :  201  (27.8%)
    three_questions_missing_fp         :  131  (18.1%)
    three_questions_missing_fn         :  117  (16.2%)
    three_questions_missing_tp         :  113  (15.7%)
    three_questions_missing_tn         :   99  (13.7%)
    two_questions                      :   31  (4.3%)
    single_question                    :   30  (4.2%)

  ARMED (n=535 with data):
    no_questions                       :  187  (25.9%)
    three_questions_missing_fn         :  129  (17.9%)
    three_questions_missing_fp         :  117  (16.2%)
    three_questions_missing_tp         :  115  (15.9%)
    three_questions_missing_tn         :  115  (15.9%)
    two_questions                      :   34  (4.7%)
    single_question                    :   25  (3.5%)

  CONV (n=536 with data):
    no_questions                       :  186  (25.8%)
    three_quest

### SDT Parameter Summary (d' and c by Theme)

In [19]:
print("--- d' (SENSITIVITY) BY THEME ---")
dprime_summary = []
for theme in THEMES:
    col = f'{theme}_d_prime'
    if col in df.columns:
        vals = df[col].dropna()
        if len(vals) > 0:
            dprime_summary.append({
                'Theme': theme, 'N': len(vals),
                'Mean': vals.mean(), 'SD': vals.std(),
                'Median': vals.median(), 'Min': vals.min(), 'Max': vals.max()
            })
if dprime_summary:
    dprime_df = pd.DataFrame(dprime_summary)
    print(dprime_df.to_string(index=False, float_format='{:.2f}'.format))

print("\n--- c (CRITERION) BY THEME ---")
c_summary = []
for theme in THEMES:
    col = f'{theme}_c'
    if col in df.columns:
        vals = df[col].dropna()
        if len(vals) > 0:
            c_summary.append({
                'Theme': theme, 'N': len(vals),
                'Mean': vals.mean(), 'SD': vals.std(),
                'Median': vals.median(), 'Min': vals.min(), 'Max': vals.max()
            })
if c_summary:
    c_df = pd.DataFrame(c_summary)
    print(c_df.to_string(index=False, float_format='{:.2f}'.format))

# By party for politically valenced themes
print("\n--- d' BY PARTY (Politically Valenced Themes) ---")
if 'party_id_7' in df.columns:
    party_groups = {'Democrat': [1,2,3], 'Independent': [4], 'Republican': [5,6,7]}
    pol_themes = [t for t, d in POLITICAL_DIRECTION.items() if d != 0]
    for theme in pol_themes:
        col = f'{theme}_d_prime'
        if col in df.columns:
            print(f"\n  {theme.upper()}:")
            for party, vals in party_groups.items():
                mask = df['party_id_7'].isin(vals)
                data = df.loc[mask, col].dropna()
                if len(data) > 0:
                    print(f"    {party:15s}: n={len(data):3d}, mean d'={data.mean():+.2f}, sd={data.std():.2f}")

--- d' (SENSITIVITY) BY THEME ---
  Theme   N  Mean   SD  Median   Min  Max
disease 476  2.23 1.86    2.56 -4.65 6.18
  armed 494  2.26 1.98    2.52 -6.18 6.18
   conv 503  1.72 1.75    1.79 -6.18 6.18
welfare 531  1.41 1.83    1.49 -6.18 6.18
   immi 504  1.56 1.79    1.67 -6.18 6.18
   vote 496  2.23 2.12    2.36 -4.05 6.18
    air 521  1.65 1.99    1.75 -6.18 6.18
firearm 514  1.95 1.80    1.97 -3.19 6.18
   auto 509  1.82 1.83    1.77 -4.95 6.18

--- c (CRITERION) BY THEME ---
  Theme   N  Mean   SD  Median   Min  Max
disease 476  0.10 0.61    0.00 -2.80 3.09
  armed 494  0.15 0.63    0.13 -3.09 3.09
   conv 503 -0.01 0.56   -0.00 -3.09 3.09
welfare 531  0.11 0.68    0.00 -3.09 3.09
   immi 504  0.04 0.74   -0.00 -2.37 3.09
   vote 496  0.27 0.86    0.09 -2.80 3.09
    air 521  0.05 0.76   -0.00 -3.09 3.09
firearm 514  0.34 0.68    0.22 -3.09 3.09
   auto 509  0.12 0.60    0.00 -3.09 3.09

--- d' BY PARTY (Politically Valenced Themes) ---

  ARMED:
    Democrat       : n=264, mean 

## 14. Final Export

In [20]:
# Drop intermediate/raw columns that are no longer needed
raw_drop_patterns = [
    'news_raw', 'income_raw',
]
raw_drops = [c for c in df.columns if any(c.startswith(p) for p in raw_drop_patterns)]
df = df.drop(columns=raw_drops, errors='ignore')

# Drop incomplete participants before saving
n_before = len(df)
df = df[~df['is_incomplete']].copy()
print(f"Dropped {n_before - len(df)} incomplete participants. Remaining: {len(df)}")

# Summary
print(f"Final dataset: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nKey column groups:")

col_groups = {
    'Demographics': ['age','gender_clean','education_num','race_clean','hispanic','income_num','mobile_bin'],
    'Political': ['ideology_num','party_id_7','news_num','p_interest_num'],
    'Knowledge/Numeracy': ['pknow_score','num_correct'],
    'Risk': ['risk_taking_mean','risk_crra_estimate','risk_switch_point'],
    'Policy Preferences': ['policy_conservatism','civlib_scale','hawk_scale'],
    'Experimental': ['order','B_before_T','BT_before_P'],
    'Theme Order': ['theme_order'] + [f'theme_{t}_position' for t in THEMES],
    'SDT Aggregate': ['total_themes_answered','total_questions_answered','themes_with_sdt','themes_with_consistency'],
    'Error Threshold Summary': ['n_err_themes_assigned','n_err_valid','err_log_ratio_mean'],
    'Quality': ['is_incomplete','low_quality','passes_attention','n_attention_flags'] + [f'flag_{x}' for x in ['hl_irrational','acceleration','recaptcha','too_fast','too_slow']],
}

for group, cols in col_groups.items():
    present = [c for c in cols if c in df.columns]
    print(f"  {group}: {len(present)}/{len(cols)} columns present")

# Save
df.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved to: {OUTPUT_PATH}")

Dropped 52 incomplete participants. Remaining: 670
Final dataset: 670 rows × 541 columns

Key column groups:
  Demographics: 7/7 columns present
  Political: 4/4 columns present
  Knowledge/Numeracy: 2/2 columns present
  Risk: 3/3 columns present
  Policy Preferences: 3/3 columns present
  Experimental: 3/3 columns present
  Theme Order: 10/10 columns present
  SDT Aggregate: 4/4 columns present
  Error Threshold Summary: 3/3 columns present
  Quality: 9/9 columns present

Saved to: /content/drive/MyDrive/Projects/SDT/Data/cleaned_data.csv


## Quick Summary Check

In [21]:
print(f"N = {len(df)}")
print(f"N passing attention = {df['passes_attention'].sum()}")
print(f"\nAge: mean={df['age'].mean():.1f}, sd={df['age'].std():.1f}, range=[{df['age'].min():.0f}, {df['age'].max():.0f}]")
print(f"\nOrder conditions:")
if 'order' in df.columns:
    print(df['order'].value_counts().to_string())
print(f"\nMean themes answered: {df['total_themes_answered'].mean():.2f}")
print(f"Mean themes with SDT computable: {df['themes_with_sdt'].mean():.2f}")
print(f"Mean themes with consistency checkable: {df['themes_with_consistency'].mean():.2f}")
print(f"\nPolicy conservatism: mean={df['policy_conservatism'].mean():.2f}, sd={df['policy_conservatism'].std():.2f}")
print(f"Error log_ratio mean: mean={df['err_log_ratio_mean'].mean():.2f}, sd={df['err_log_ratio_mean'].std():.2f}")

N = 670
N passing attention = 649

Age: mean=38.2, sd=12.8, range=[18, 79]

Order conditions:
order
PBT    191
BTP    168
TBP    163
PTB    148

Mean themes answered: 7.00
Mean themes with SDT computable: 6.71
Mean themes with consistency checkable: 6.64

Policy conservatism: mean=2.58, sd=0.67
Error log_ratio mean: mean=0.09, sd=0.68
